DATA PREPARATION

This notebook prepares the data for modelling. The setup below (imports, loading and cleaning) is the same as in `01_exploratory_data_analysis.ipynb`. See that notebook for the data dictionary and the EDA findings that the decisions here are based on.

IMPORTING PACKAGES

In [5]:
import numpy as np # linear algebra
import pandas as pd # data processing
from sklearn.impute import SimpleImputer
from matplotlib import pyplot as plt #data visualization
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
from sklearn.base import clone
import joblib
from pathlib import Path

LOADING DATA

Load the data of the csv file containing Netflix's customer churn information.
The file contains attributes such as customer ID, age, gender, subscription type, watch hours, last login date and more.    

In [6]:
X = pd.read_csv('/Users/wiegerhelsdingen/Documents/Github/netflix-churn-prediction/src/netflix_customer_churn.csv')

CLEANING DATA

First separate target from the dataset (y).
Then we'll only keep the selected features: 'subscription_type', 'payment_method', 'last_login_days', 'number_of_profiles', 'avg_watch_time_per_day'.
Then we split the data into numerical and categorical values.

In [7]:
# Check for missing values
missing_values = X.isnull().sum()
print('Missing values in each column:')
print(missing_values)

# Data Types
print('\nData types:')
print(X.dtypes)


# Remove rows with missing target, separate target from predictors
X.dropna(axis=0, subset=['churned'], inplace=True)
y = X.churned
X.drop(['churned'], axis=1, inplace=True)

# remove duplicate rows
X = X.drop_duplicates()

# Split the dataset into categorical and numeric columns
categorical_columns = ['subscription_type', 'payment_method']
for col in categorical_columns:
    X[col] = X[col].astype('category')
X_cat = X[categorical_columns].copy()

print('\nCategorical columns:')
print(categorical_columns)

numeric_columns = ['last_login_days', 'number_of_profiles', 'avg_watch_time_per_day']
uniform_columns = ['last_login_days', 'number_of_profiles']
skewed_columns = ['avg_watch_time_per_day']
for col in numeric_columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')
X_num = X[numeric_columns].copy()

print('\nNumeric columns:')
print(numeric_columns)



Missing values in each column:
customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
dtype: int64

Data types:
customer_id                   str
age                         int64
gender                        str
subscription_type             str
watch_hours               float64
last_login_days             int64
region                        str
device                        str
monthly_fee               float64
churned                     int64
payment_method                str
number_of_profiles          int64
avg_watch_time_per_day    float64
favorite_genre                str
dtype: object

Categorical columns:
['subscription_type', 'payment_method']

TRAIN/TEST SPLIT

We split the data **before** any preprocessing. The imputers and the scaler learn values from the data (means, most frequent categories, standard deviations). If they learn them from all rows, information from the test set leaks into training, and the test score becomes too optimistic.

- `test_size=0.2` 80% train, 20% test
- `stratify=y` keeps the churn ratio (~50/50) the same in both sets
- `random_state=42` makes the split reproducible

In [8]:
# Keep only the selected features
X = X[numeric_columns + categorical_columns]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Train: {X_train.shape[0]} rows, churn rate {y_train.mean():.1%}')
print(f'Test:  {X_test.shape[0]} rows, churn rate {y_test.mean():.1%}')

Train: 4000 rows, churn rate 50.3%
Test:  1000 rows, churn rate 50.3%


PREPROCESSING PIPELINE

Instead of running each step by hand, we put them in a scikit-learn pipeline.
We'll use a different pipeline for each preprocessing method:


| Pipeline | Columns | Steps | Why |
|---|---|---|---|
| **Uniform** | `last_login_days`, `number_of_profiles` | median imputer → min-max scaling (0 to 1) | Bounded, evenly spread, no outliers, so linear scaling is enough |
| **Skewed** | `avg_watch_time_per_day` | median imputer → `log1p` → standardize (mean 0, std 1) | Most values are below 1 but a few go up to 98. The log fixes the shape, the scaler puts it on the same footing as the other features |
| **Categorical** | `subscription_type`, `payment_method` | most-frequent imputer → one-hot encoding | Models need numbers, not text |

A `ColumnTransformer` sends each group of columns to its own pipeline and combines the results.

We considered clipping `avg_watch_time_per_day` at 40, but it only affects 2 rows and doesn't fix the skew. The log does both: it pulls in the outliers and spreads out the crowded low values, where churn changes most (around 0.5 hours per day).

`log1p` means log(1 + x). We use it instead of log(x) because some customers have a value of 0, and log(0) is undefined.

Why a pipeline?
1. **No leakage:** `fit` only ever sees the training data; the test data is only `transform`ed with what was learned from training.
2. **The same steps every time:** new customers get exactly the same treatment, which we need for the prediction API later.
3. **Less code:** in model development we attach the model to the end of this pipeline and train everything in one go.

There are no missing values in this dataset, but the imputers protect us if new data has gaps.
`handle_unknown='ignore'` means a category we never saw in training (for example a new payment method) becomes all zeros instead of an error.

In [9]:
# Pipeline for uniform numeric features: impute missing values with the median and scale to [0, 1].
uniform_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

# Pipeline for skewed numeric features: impute missing values with the median, log-transform, and standardize.
skewed_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

# Pipeline for categorical features: impute missing values with the most frequent value and one-hot encode.
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Sends each group of columns to its own pipeline and combines the results.
preprocessor = ColumnTransformer(transformers=[
    ('uniform', uniform_pipeline, uniform_columns),
    ('skewed', skewed_pipeline, skewed_columns),
    ('cat', categorical_pipeline, categorical_columns)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('uniform', ...), ('skewed', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

EXPERIMENT SETUP

All models are compared in the same way, using **5-fold cross-validation on the training set only**. The test set stays untouched until the final evaluation.

- **Cross validation** splits the training set into 5 folds with the same churn ratio.
- **Selection metric** ROC AUC as it gives insight into how well positives rank higher than negatives, especially useful in a probability prediction (0-1). Then recall since I'd like to be more strict on false negatives. Then accuracy as extra insight, given the target distribution is balanced.

The `evaluate` helper runs cross-validation for a model and adds a row to the `results` table, so every experiment is one line of code and all results end up in one place. The `± std` columns show how much the score varies between folds: a small spread means the result is stable.

In [10]:
# Make sure each fold has same distribution of the target variable
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
evaluation_metrics = ['roc_auc', 'recall', 'accuracy']
results = {}

def evaluate(model_pipeline, name):
    # Cross-validate a model on the training set and add its scores to the results table."""
    scores = cross_validate(model_pipeline, X_train, y_train, cv=cv, scoring=evaluation_metrics)
    scores_mean = {metric: round(float(scores[f'test_{metric}'].mean()), 3) for metric in evaluation_metrics}
    model_pipeline_name = type(model_pipeline.steps[-1][1]).__name__
    results[name] = scores_mean
    return name, scores_mean

def show_results():
    return pd.DataFrame(results).T.sort_values('roc_auc', ascending=False).round(3)

BASELINE MODELS

**Dummy model.** `DummyClassifier` ignores the features and always predicts the most frequent class (churn, at 50.3%). It's the "zero line" every real model has to beat. Note that it scores a recall of 1.0: by predicting that *everyone* churns, it catches every churner. That's why recall can never be the only metric.

**Logistic regression.** A simple, interpretable model, and our real baseline. It uses the preprocessing pipeline from above: min-max scaling for the uniform columns, log + standardizing for the skewed column, and one-hot encoding for the categorical columns. `max_iter=1000` gives the solver enough steps to converge.

In [11]:
dummy_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])
evaluate(dummy_model, 'dummy_model')

('dummy_model', {'roc_auc': 0.5, 'recall': 1.0, 'accuracy': 0.503})

In [12]:
logreg_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        C=1.0,               # regularization strength (inverse: smaller C = stronger penalty)
        solver='lbfgs',      # optimization algorithm, the default and fine for small data
        max_iter=1000,       # enough iterations to converge
        random_state=42,     # only used by some solvers, but harmless and explicit
    ))
])
evaluate(logreg_model, 'logreg_model')

('logreg_model', {'roc_auc': 0.969, 'recall': 0.917, 'accuracy': 0.897})

In [13]:
# Select a default XGBoost model with some reasonable hyperparameters to see if it performs better than baseline
# We don't use early stopping here due to cross-validation, also not necessary for a small dataset, but useful for larger datasets.


xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=100,    # number of trees in the ensemble
        max_depth=3,         # maximum depth of each tree
        learning_rate=0.1,   # step size shrinkage used in update to prevent overfitting
        random_state=42,     # for reproducibility
        # use_label_encoder=False,  # avoid warning about label encoding
        eval_metric='logloss'      # evaluation metric for binary classification
    ))
])
evaluate(xgb_model, 'xgb_default')

show_results()

,roc_auc,recall,accuracy
xgb_default,0.995,0.965,0.969
logreg_model,0.969,0.917,0.897
dummy_model,0.500,1.000,0.503


XGBoost scores much higher than the baseline model making it worthwhile to move forward with this model.
Now, to tweak some parameters, we'll pick a few randomized parameter combinations and perform cross validation on them.
This will give us an insight in the most optimal combinations.

In [14]:
param_distributions = {
    'classifier__n_estimators':     randint(100, 1000),
    'classifier__learning_rate':    uniform(0.01, 0.2),
    'classifier__max_depth':        randint(2, 7), 
    # preprocessing parameters work the same way
    'preprocessor__uniform__imputer__strategy': ['median', 'mean'],
}

search = RandomizedSearchCV(
    xgb_model, param_distributions,
    n_iter=40, 
    scoring=evaluation_metrics, 
    refit='roc_auc', # pick the winner based on ROC AUC score
    cv=cv, 
    random_state=42,
    n_jobs=-1
)

# Run cross validation on all XGboost parameter combinations
search.fit(X_train, y_train)

# the winning hyperparameters and their scores
print(search.best_params_)         
print(round(search.best_score_, 3))

# Add the tuned model to your results table
evaluate(search.best_estimator_, 'xgb_tuned')
show_results()

{'classifier__learning_rate': np.float64(0.16593820005455387), 'classifier__max_depth': 6, 'classifier__n_estimators': 714, 'preprocessor__uniform__imputer__strategy': 'mean'}
0.999


,roc_auc,recall,accuracy
xgb_tuned,0.999,0.979,0.981
xgb_default,0.995,0.965,0.969
logreg_model,0.969,0.917,0.897
dummy_model,0.500,1.000,0.503


DATA LEAKAGE CHECK

We'll check again if 'last_login_days' contains any information about a churning customer that we don't already have at prediction time. During EDA we saw no clear leakage signs, only strong correlation.

2 questions to answer here:
- Is the feature known before churn happens?
- How much do you lose without it? 

We see that removing 'last_login_days' lowers the ROC AUC with 0.031 (or 3-4%), which could be significant in big retention campaigns. Since there is no clear proof of leakage, and we expect this feature to be available at prediction time, we'll keep it. But we'll keep an eye on leakage during monitoring. 

In [15]:

def make_preprocessor(uniform_cols, skewed_cols, cat_cols=categorical_columns):
    return ColumnTransformer([
        ('uniform', uniform_pipeline, uniform_cols),
        ('skewed', skewed_pipeline, skewed_cols),
        ('cat', categorical_pipeline, cat_cols),
    ])

num_feature_sets = {
    'all features':       (['last_login_days', 'number_of_profiles'], ['avg_watch_time_per_day']),
    'no last_login_days': (['number_of_profiles'],                    ['avg_watch_time_per_day']),
}

for label, (uniform_cols, skewed_cols) in num_feature_sets.items():
    model = clone(search.best_estimator_).set_params(
        preprocessor=make_preprocessor(uniform_cols, skewed_cols)
    )
    evaluate(model, f'XGBoost tuned – {label}')

show_results()

,roc_auc,recall,accuracy
xgb_tuned,0.999,0.979,0.981
XGBoost tuned – all features,0.999,0.979,0.981
xgb_default,0.995,0.965,0.969
logreg_model,0.969,0.917,0.897
XGBoost tuned – no last_login_days,0.968,0.901,0.895
dummy_model,0.500,1.000,0.503


In [16]:
Path('../models').mkdir(exist_ok=True)
final_model = search.best_estimator_ 
joblib.dump(final_model, '../models/final_model.joblib')

['../models/final_model.joblib']